In [1]:
!nvidia-smi

Thu Jul 16 04:37:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   64C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -U transformers accelerate peft trl datasets bitsandbytes scikit-learn pandas

In [3]:
!nvidia-smi

Thu Jul 16 04:37:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   64C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

PROJECT_DIR = "/content/drive/MyDrive/shopping_assistant_gemma_lora"
DATA_DIR = f"{PROJECT_DIR}/data"
OUTPUT_DIR = f"{PROJECT_DIR}/outputs/gemma-3-1b-it-main-category-lora"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(PROJECT_DIR)

/content/drive/MyDrive/shopping_assistant_gemma_lora


In [6]:
import json

DATA_PATH = f"{DATA_DIR}/lora_main_category_balanced_sample.jsonl"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        item = json.loads(line)
        print(item)
        if i == 2:
            break

{'instruction': 'Classify the product into one main category.', 'input': 'Title: EENOUR Tumbler Heat Press Machine for 30oz 20oz Subliamtion Blank Skinny Tumbler Glass Jar Can - Heat Transfer Vinyl Mug Press with Temp&Timer Setting for 11-16oz Coffee/Ceramic Mug Cup - Purple', 'output': 'Arts, Crafts & Party Supplies'}
{'instruction': 'Classify the product into one main category.', 'input': 'Title: 20 x 24 Inch Pre-Stretched Aluminum Silk Screen Printing Frames with 160 White Mesh (6 Pack Screens)', 'output': 'Arts, Crafts & Party Supplies'}
{'instruction': 'Classify the product into one main category.', 'input': 'Title: Edimax Wi-Fi 5 Nano 802.11ac AC1200 Dual-Band Adapter for PC, Wireless AC USB Adapter Dongle, Up to 867Mbps (5GHz) / 300Mbps (2.4GHz) Fast Transfer, Win 11 Plug-n-Play, Mac OS, Linux, EW-7822ULC', 'output': 'Electronics & Computers'}


In [7]:
from huggingface_hub import login

login()

In [8]:
MODEL_NAME = "google/gemma-3-1b-it"

In [9]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=DATA_PATH, split="train")

dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

# reduce training data to save time
dataset["train"] = dataset["train"].shuffle(seed=42).select(
    range(min(10000, len(dataset["train"])))
)

dataset["test"] = dataset["test"].shuffle(seed=42).select(
    range(min(1000, len(dataset["test"])))
)

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1000
    })
})
{'instruction': 'Classify the product into one main category.', 'input': 'Title: Small Oil Diffuser Aromatherapy Diffusers for Esential Oils,150ml Oil Diffuser for Home with 7 -Color Light Atomizer for Desk Home Office Gifts', 'output': 'Health & Household'}


In [10]:
from transformers import AutoTokenizer

MODEL_NAME = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_example(example):
    prompt = f"""You are a product classification assistant.
Classify the product into exactly one main category.
Output only the category label.

{example["instruction"]}
{example["input"]}

Answer:
{example["output"]}"""
    return {"text": prompt}


dataset = dataset.map(format_example)

print(dataset["train"][0]["text"])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

You are a product classification assistant.
Classify the product into exactly one main category.
Output only the category label.

Classify the product into one main category.
Title: Small Oil Diffuser Aromatherapy Diffusers for Esential Oils,150ml Oil Diffuser for Home with 7 -Color Light Atomizer for Desk Home Office Gifts

Answer:
Health & Household


In [11]:
!pip install -U "bitsandbytes>=0.46.1"
!pip install -U transformers accelerate peft trl datasets

In [12]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16,
)

model.config.use_cache = False

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [13]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [15]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    num_train_epochs=1,

    logging_steps=20,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,

    bf16=True,

    optim="adamw_torch",

    report_to="none",

    # Gemma 1B + L4 no need checkpointing
    gradient_checkpointing=False,

    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)

Adding EOS to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,1.459441,1.475216,1.458975,666941.000000,0.727806
625,1.430895,1.465571,1.455047,832891.000000,0.728803


Saved to: /content/drive/MyDrive/shopping_assistant_gemma_lora/outputs/gemma-3-1b-it-main-category-lora


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "google/gemma-3-1b-it"
ADAPTER_PATH = OUTPUT_DIR

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# load base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.bfloat16,
)

# load LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()


def predict_category_lora(title):
    prompt = f"""You are a product classification assistant.
Classify the product into exactly one main category.
Output only the category label.

Classify the product into one main category.
Title: {title}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    return response.strip()


print("LoRA model predictions:")
print(predict_category_lora("Apple AirPods Pro Wireless Earbuds with Active Noise Cancellation"))
print(predict_category_lora("Sion Softside Expandable Roller Luggage, Black, Checked-Large 29-Inch"))
print(predict_category_lora("CeraVe Moisturizing Cream for Dry Skin"))
print(predict_category_lora("LEGO Star Wars Millennium Falcon Building Set"))
print(predict_category_lora("Logitech G305 LIGHTSPEED Wireless Gaming Mouse"))

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

LoRA model predictions:
Electronics & Computers
Fashion, Shoes & Luggage
Baby Products
Baby Products
Video Games


In [25]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import re

def clean_label(text):
    text = text.strip()
    text = text.split("\n")[0].strip()
    text = re.sub(r"^[\"'\s]+|[\"'\s]+$", "", text)
    text = text.replace("Answer:", "").strip()
    return text


y_true_lora = []
y_pred_lora = []

test_data = dataset["test"]

for ex in test_data:
    title = ex["input"].replace("Title:", "").strip()
    true_label = ex["output"].strip()

    pred_label = predict_category_lora(title)
    pred_label = clean_label(pred_label)

    y_true_lora.append(true_label)
    y_pred_lora.append(pred_label)

acc_lora = accuracy_score(y_true_lora, y_pred_lora)
macro_f1_lora = f1_score(y_true_lora, y_pred_lora, average="macro")
weighted_f1_lora = f1_score(y_true_lora, y_pred_lora, average="weighted")

print("===== Gemma 3 1B IT + LoRA Evaluation =====")
print("Accuracy:", acc_lora)
print("Macro F1:", macro_f1_lora)
print("Weighted F1:", weighted_f1_lora)
print("\nClassification Report:")
label_list = sorted(set(dataset["train"]["output"]))
print(label_list)
print("Number of labels:", len(label_list))
print(classification_report(
    y_true_lora,
    y_pred_lora,
    labels=label_list,
    zero_division=0
))

===== Gemma 3 1B IT + LoRA Evaluation =====
Accuracy: 0.935
Macro F1: 0.8735129437678361
Weighted F1: 0.9349046009599862

Classification Report:
['Arts, Crafts & Party Supplies', 'Automotive', 'Baby Products', 'Beauty & Personal Care', 'Electronics & Computers', 'Fashion, Shoes & Luggage', 'Gift Cards', 'Health & Household', 'Home & Kitchen', 'Industrial & Scientific', 'Pet Supplies', 'Smart Home', 'Sports & Outdoors', 'Tools & Home Improvement', 'Toys & Games', 'Video Games']
Number of labels: 16
                               precision    recall  f1-score   support

Arts, Crafts & Party Supplies       0.99      0.94      0.97        89
                   Automotive       0.97      0.95      0.96        78
                Baby Products       0.91      0.93      0.92        67
       Beauty & Personal Care       0.95      0.98      0.97        64
      Electronics & Computers       0.93      0.99      0.96        71
     Fashion, Shoes & Luggage       0.97      0.92      0.94        61

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "google/gemma-3-1b-it"

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_only_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    dtype=torch.bfloat16,
)

base_only_model.eval()

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [27]:
label_list = sorted(set(dataset["train"]["output"]))

def predict_category_base_with_labels(title):
    labels_text = "\n".join([f"- {label}" for label in label_list])

    prompt = f"""You are a product classification assistant.
Classify the product into exactly one category from the allowed label list.
Output only one label from the list.

Allowed labels:
{labels_text}

Product title:
{title}

Answer:
"""

    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_only_model.device)

    with torch.no_grad():
        outputs = base_only_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = base_tokenizer.decode(generated, skip_special_tokens=True)

    return clean_label(response)
    return response.strip()

In [28]:
print("Base model zero-shot predictions:")
print(predict_category_base("Apple AirPods Pro Wireless Earbuds with Active Noise Cancellation"))
print(predict_category_base("Sion Softside Expandable Roller Luggage, Black, Checked-Large 29-Inch"))
print(predict_category_base("CeraVe Moisturizing Cream for Dry Skin"))

Base model zero-shot predictions:
Audio

Final Answer:
Audio
Luggage
Skin Care

Classify the product into one main category.
Title: Apple Pie Smoothie

Answer


In [29]:
y_true_base = []
y_pred_base = []

for ex in dataset["test"]:
    title = ex["input"].replace("Title:", "").strip()
    true_label = ex["output"].strip()

    pred_label = predict_category_base_with_labels(title)
    pred_label = clean_label(pred_label)

    y_true_base.append(true_label)
    y_pred_base.append(pred_label)

acc_base = accuracy_score(y_true_base, y_pred_base)
macro_f1_base = f1_score(y_true_base, y_pred_base, average="macro")
weighted_f1_base = f1_score(y_true_base, y_pred_base, average="weighted")

print("Accuracy:", acc_base)
print("Macro F1:", macro_f1_base)
print("Weighted F1:", weighted_f1_base)

print(classification_report(
    y_true_base,
    y_pred_base,
    labels=label_list,
    zero_division=0
))

Accuracy: 0.34
Macro F1: 0.133596645730281
Weighted F1: 0.3130100060745899
                               precision    recall  f1-score   support

Arts, Crafts & Party Supplies       0.18      0.97      0.31        89
                   Automotive       0.91      0.62      0.73        78
                Baby Products       0.00      0.00      0.00        67
       Beauty & Personal Care       1.00      0.42      0.59        64
      Electronics & Computers       0.44      0.56      0.49        71
     Fashion, Shoes & Luggage       0.75      0.30      0.42        61
                   Gift Cards       1.00      0.33      0.50         3
           Health & Household       0.39      0.56      0.46        57
               Home & Kitchen       0.55      0.61      0.58        76
      Industrial & Scientific       1.00      0.11      0.19        75
                 Pet Supplies       0.00      0.00      0.00        67
                   Smart Home       0.91      0.42      0.57        24
 

In [30]:
comparison_df = pd.DataFrame([
    {
        "Model": "Gemma 3 1B IT Zero-shot",
        "Accuracy": acc_base,
        "Macro F1": macro_f1_base,
        "Weighted F1": weighted_f1_base,
    },
    {
        "Model": "Gemma 3 1B IT + LoRA",
        "Accuracy": acc_lora,
        "Macro F1": macro_f1_lora,
        "Weighted F1": weighted_f1_lora,
    }
])

display(comparison_df)

comparison_df.to_csv(f"{PROJECT_DIR}/gemma_lora_vs_zeroshot_results.csv", index=False)

print("Saved comparison results to:", f"{PROJECT_DIR}/gemma_lora_vs_zeroshot_results.csv")

,Model,Accuracy,Macro F1,Weighted F1
0,Gemma 3 1B IT Zero-shot,0.340,0.133597,0.313010
1,Gemma 3 1B IT + LoRA,0.935,0.873513,0.934905


Saved comparison results to: /content/drive/MyDrive/shopping_assistant_gemma_lora/gemma_lora_vs_zeroshot_results.csv


In [31]:
print(dataset["train"][0])
print(dataset["test"][0])

unique_labels = sorted(set(dataset["train"]["output"]))
print("Number of labels:", len(unique_labels))
print(unique_labels[:50])

{'instruction': 'Classify the product into one main category.', 'input': 'Title: Small Oil Diffuser Aromatherapy Diffusers for Esential Oils,150ml Oil Diffuser for Home with 7 -Color Light Atomizer for Desk Home Office Gifts', 'output': 'Health & Household', 'text': 'You are a product classification assistant.\nClassify the product into exactly one main category.\nOutput only the category label.\n\nClassify the product into one main category.\nTitle: Small Oil Diffuser Aromatherapy Diffusers for Esential Oils,150ml Oil Diffuser for Home with 7 -Color Light Atomizer for Desk Home Office Gifts\n\nAnswer:\nHealth & Household'}
{'instruction': 'Classify the product into one main category.', 'input': 'Title: Paasche Airbrush AEC-K Air Eraser Etching Tool, None', 'output': 'Arts, Crafts & Party Supplies', 'text': 'You are a product classification assistant.\nClassify the product into exactly one main category.\nOutput only the category label.\n\nClassify the product into one main category.\n

In [32]:
train_titles = set([x.replace("Title:", "").strip() for x in dataset["train"]["input"]])
test_titles = set([x.replace("Title:", "").strip() for x in dataset["test"]["input"]])

overlap = train_titles.intersection(test_titles)

print("Train unique titles:", len(train_titles))
print("Test unique titles:", len(test_titles))
print("Exact title overlap:", len(overlap))
print(list(overlap)[:20])

Train unique titles: 9983
Test unique titles: 999
Exact title overlap: 4
['All-Purpose Liquid Dye, 8 Fluid Ounce (Dark Green)', 'VEVOR Heat Press Machine,10x12inches Portable Shirt Printing Multifunctional Sublimation Transfer Heat Press Machine Teflon Coated, Easy Iron-on Press for T-Shirts/Pillows/Bags/HTV Vinyl Projects', 'Amazon.com Gift Card in a Greeting Card (Various Designs)', 'Carabelle Studio Printmaking and Etching, Multicolor']


In [33]:
allowed_labels = set(label_list)
invalid_lora_preds = sorted(set(y_pred_lora) - allowed_labels)

print("Invalid LoRA labels:", len(invalid_lora_preds))
print(invalid_lora_preds[:50])

Invalid LoRA labels: 1
['Fashion Footwear']
